In [1]:
from pynq import Overlay, allocate   # Overlay = bitstream + metadata, allocate = DMA-capable memory
import numpy as np, time            # array math and the reference FFT

ol = Overlay("dft.bit")              # programs the FPGA with dft.bit matching dft.hwh
print(ol.ip_dict.keys())             # print names of the IP blocks found in the .hwh.

dict_keys(['dft_0', 'axi_dma_0', 'axi_dma_1', 'processing_system7_0'])


In [2]:
dma_r, dma_i = ol.axi_dma_0, ol.axi_dma_1
dft = ol.dft_0
SIZE = 1024
in_r  = allocate(shape=(SIZE,), dtype=np.float32)   # real input   -> DMA 0 send
in_i  = allocate(shape=(SIZE,), dtype=np.float32)   # imag input   -> DMA 1 send
out_r = allocate(shape=(SIZE,), dtype=np.float32)   # real output  <- DMA 0 receive
out_i = allocate(shape=(SIZE,), dtype=np.float32)   # imag output  <- DMA 1 receive
in_r[:] = np.arange(SIZE); # test signal: x[n] = n (real) + 0j. The first output bin is the sum 0+1+...+1023 = 523776
in_i[:] = 0          
out_r[:] = 0; out_i[:] = 0

def status(tag):
    print(f"{tag:8s} ctrl={dft.read(0x00):#x} | "
          f"real MM2S={dma_r.mmio.read(0x04):#x} S2MM={dma_r.mmio.read(0x34):#x} | "
          f"imag MM2S={dma_i.mmio.read(0x04):#x} S2MM={dma_i.mmio.read(0x34):#x}")

status("before")
dft.write(0x00, 0x01)                       # AP_START
dma_r.recvchannel.transfer(out_r)
dma_i.recvchannel.transfer(out_i)
dma_r.sendchannel.transfer(in_r)
dma_i.sendchannel.transfer(in_i)

t0 = time.time()
while time.time() - t0 < 3:
    done = all(r & 0x1000 for r in (dma_r.mmio.read(0x04), dma_r.mmio.read(0x34),
                                    dma_i.mmio.read(0x04), dma_i.mmio.read(0x34)))
    if done: break
    time.sleep(0.05)
status("after")
print("completed" if done else "TIMED OUT", f"{(time.time()-t0)*1000:.1f} ms")

before   ctrl=0x4 | real MM2S=0x0 S2MM=0x0 | imag MM2S=0x0 S2MM=0x0
after    ctrl=0xe | real MM2S=0x1002 S2MM=0x1002 | imag MM2S=0x1002 S2MM=0x1002
completed 53.4 ms


In [3]:
# ---- check against NumPy---------------
ref = np.fft.fft(in_r + 1j*in_i)
print("RMSE real:", np.sqrt(np.mean((out_r - ref.real)**2)))
print("RMSE imag:", np.sqrt(np.mean((out_i - ref.imag)**2)))
print("first 3 hw :", out_r[:3], out_i[:3])
print("first 3 ref:", ref.real[:3], ref.imag[:3])

RMSE real: 0.0019347736883832997
RMSE imag: 0.00858506378907264
first 3 hw : [ 5.2377600e+05 -5.1200800e+02 -5.1199518e+02] [     0.   166885.69  83441.93]
first 3 ref: [ 5.23776e+05 -5.12000e+02 -5.12000e+02] [     0.         166885.53000842  83441.97960358]


In [4]:

def run():
    dft.write(0x00, 0x01)
    dma_r.recvchannel.transfer(out_r); dma_i.recvchannel.transfer(out_i)
    dma_r.sendchannel.transfer(in_r);  dma_i.sendchannel.transfer(in_i)
    dma_r.sendchannel.wait(); dma_i.sendchannel.wait()
    dma_r.recvchannel.wait(); dma_i.recvchannel.wait()

N = 100          # average over many runs;
t = time.time()
for _ in range(N): run()
dt = (time.time() - t) / N

ref = np.fft.fft(in_r + 1j*in_i)
print("RMSE real:", np.sqrt(np.mean((out_r - ref.real)**2)))
print("RMSE imag:", np.sqrt(np.mean((out_i - ref.imag)**2)))
print(f"{dt*1000:.3f} ms per DFT1024  ->  {1/dt:.0f} DFT1024/s (hardware + DMA + Python)") # 1.332 ms is the accelerator-only estimate from synthesis: 133,152 cycles at 100 MHz.
                                                                 # measured time is larger because every call also pays for starting and waiting on
print(f"HLS estimate: {1.33153:.3f} ms  ->  751 DFT1024/s")     # four DMA transfers from Python

RMSE real: 0.0019347736883832997
RMSE imag: 0.00858506378907264
3.560 ms per DFT1024  ->  281 DFT1024/s (hardware + DMA + Python)
HLS estimate: 1.332 ms  ->  751 DFT1024/s


In [ ]:
#----------------debugging only---------------
#def status(tag):
#    print(f"{tag:8s} ctrl={dft.read(0x00):#x} | "
#          f"real MM2S={dma_r.mmio.read(0x04):#x} S2MM={dma_r.mmio.read(0x34):#x} | "
#          f"imag MM2S={dma_i.mmio.read(0x04):#x} S2MM={dma_i.mmio.read(0x34):#x}")
#
#status("now")